In [1]:
from fastapi_offline import FastAPIOffline
import uvicorn
import asyncio

In [2]:
# Database with SQLite
import sqlite3

def connect_db():
   conn = sqlite3.connect("users.db")
   conn.row_factory = sqlite3.Row
   return conn

conn = connect_db()
cur = conn.cursor()

# Create users table if it doesn't exist
conn.execute(
  """
  CREATE TABLE IF NOT EXISTS users (
      id INTEGER PRIMARY KEY AUTOINCREMENT,
      username TEXT NOT NULL,
      password TEXT NOT NULL
  )
  """
)

conn.commit()

In [3]:
# CRUD operations

# Get all users
def get_users():
  conn = connect_db()
  cur = conn.cursor()
  rows = cur.execute("SELECT * FROM users").fetchall()
  conn.close()

  return [dict(row) for row in rows]

def get_user(user_id: int):
  conn = connect_db()
  cur = conn.cursor()
  row = cur.execute("SELECT * FROM users WHERE id = ?", (user_id,)).fetchone()
  conn.close()

  return dict(row) if row else None

def create_user(username: str, password: str):
  conn = connect_db()
  cur = conn.cursor()
  cur.execute("INSERT INTO users (username, password) VALUES (?, ?)", (username, password))
  conn.commit()
  user_id = cur.lastrowid
  conn.close()

  return get_user(user_id)

In [12]:
# App

app = FastAPIOffline()

# Rounting - กำหนดเส้นทางของ API (Controller)

# Rest API
@app.get("/")
def read_root():
    return {
        "success": True,
        "message": "Hello, World!",
        "data": None
    }

# Get all users
@app.get("/users")
def read_users():
    users = get_users()
    return users

# Get user by ID
@app.get("/users/{user_id}")
def read_user(user_id: str):
    
    res = get_user(int(user_id))
    if res:
        return res
    else:
        return {
            "success": False,
            "message": "User not found",
            "data": None
        }

# Create a new user (Test)
@app.get("/users/{username}/{password}")
def create_test_user(username: str, password: str):
    user = create_user(username, password)
    return {
        "success": True,
        "message": "User created successfully",
        "data": user
    }

In [13]:
if __name__ == "__main__":
  config = uvicorn.Config(app)
  server = uvicorn.Server(config)
  await server.serve()

INFO:     Started server process [3240]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:59132 - "GET /users/3 HTTP/1.1" 200 OK
INFO:     127.0.0.1:55271 - "GET /users/1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:55271 - "GET /users/2 HTTP/1.1" 200 OK
INFO:     127.0.0.1:55271 - "GET /users HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [3240]
